# Notebook 04 — Baseline Experiments

Runs all 5 baselines (B0–B4) on DTAH-Bench-50 and saves results.

- **B0–B3:** CPU baselines (spec generation only, no 3D mesh)
- **B4:** Full IFC-GraphRAG-DT pipeline (GPU recommended for 3D generation)

Results are saved to Google Drive for persistence across sessions.

In [1]:
import os, sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if not Path("pipeline").exists():
    if Path("../pipeline").exists():
        os.chdir("..")
    elif Path("ifc-graphrag-dt/pipeline").exists():
        os.chdir("ifc-graphrag-dt")
    elif IS_COLAB:
        !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
        !bash ifc-graphrag-dt/colab_setup.sh
        os.chdir("ifc-graphrag-dt")
    else:
        raise RuntimeError("Run this notebook from the ifc-graphrag-dt repository root.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

import json, time
import torch
from pathlib import Path
import yaml
import numpy as np
from benchmark.dtah_bench import DTAHBench
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from pipeline.baselines.b0_no_grounding import B0NoGrounding
from pipeline.baselines.b1_llm_only import B1LLMOnly
from pipeline.baselines.b2_flat_rag import B2FlatRAG
from pipeline.baselines.b3_ifc_lookup import B3IFCLookup
from evaluation.metrics.kcs_dt import KCSDTScorer

DRIVE_OUT = "outputs"
SPEC_PROVIDER = "anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "deterministic"
IFC_PATH = "benchmark/ifc_reference_models/duplex.ifc"
GRAPH_CACHE = "outputs/graphs/ifc_graph.json"
EMB_CACHE = "outputs/embedders/graph_embedder"
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Layer 2 provider: {SPEC_PROVIDER}")

for directory in ["outputs/scores", "outputs/specs", "outputs/meshes", "outputs/figures"]:
    os.makedirs(directory, exist_ok=True)
print("All imports OK")


Working directory: E:\PhD\Journal-2\Implementation\ifc-graphrag-dt


GPU available: False
Layer 2 provider: deterministic
All imports OK


In [2]:
from pipeline.ifc_assets import ensure_duplex_ifc

DUPLEX_PATH = ensure_duplex_ifc()
IFC_PATH = str(DUPLEX_PATH)
print(f"IFC file ready: {DUPLEX_PATH} ({DUPLEX_PATH.stat().st_size / 1024:.1f} KB)")

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(f'{EMB_CACHE}/faiss.index').exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embedder...')
    embedder = GraphEmbedder()
    embedder.fit(G)
    embedder.save(EMB_CACHE)

with open('pipeline/configs/ifc_config.yaml') as f:
    ifc_config = yaml.safe_load(f)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

IFC file ready: benchmark\ifc_reference_models\duplex.ifc (2325.0 KB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Graph: 296 nodes | Embedder: 296 nodes


In [3]:
# ── Cell 3: Initialise baselines ──────────────────────────────────────────────
TIER_DEPTHS = {1: 1, 2: 2, 3: 4}
llm_config  = {'llm_provider': SPEC_PROVIDER, 'model': 'claude-sonnet-4-20250514',
               'max_tokens': 2048, 'temperature': 0.0, 'output_dir': 'outputs/specs'}
gen_config  = {'output_dir': 'outputs/meshes'}

b0 = B0NoGrounding(gen_config)
b1 = B1LLMOnly(llm_config)
b2 = B2FlatRAG(llm_config, graph_embedder=embedder)
b3 = B3IFCLookup(llm_config)
b4_spec_gen = SceneSpecGenerator(llm_config)

bench       = DTAHBench(pilot_mode=True)
all_prompts = bench.load_all()
print(f'Running on {len(all_prompts)} pilot prompts across {len(set(p["tier"] for p in all_prompts))} tiers')

Running on 50 pilot prompts across 3 tiers


In [4]:
# ── Cell 4: Run B0, B1, B2, B3 (CPU baselines) ───────────────────────────────
baseline_records = {'b0': [], 'b1': [], 'b2': [], 'b3': []}

for i, p in enumerate(all_prompts):
    pid, prompt, tier, domain = p['id'], p['prompt'], p['tier'], p.get('domain','MEP')
    depth    = TIER_DEPTHS[tier]
    seeds    = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids = [s['node_id'] for s in seeds]
    trav     = KHopTraversal(G, max_depth=depth, bidirectional=True)
    graph_sg = trav.traverse(seed_ids).subgraph

    # B0: no grounding
    s0 = b0.run(prompt, pid)
    baseline_records['b0'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': 0, 'relations': 0, 'baseline': 'B0', 'status': 'ok'})

    # B1: LLM only
    try:
        s1 = b1.run(prompt, pid)
        baseline_records['b1'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
            'entities': len(s1.get('entities',[])), 'relations': len(s1.get('relations',[])),
            'baseline': 'B1', 'status': 'ok'})
    except Exception as e:
        baseline_records['b1'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
            'entities': 0, 'relations': 0, 'baseline': 'B1', 'status': f'error: {e}'})

    # B2: flat RAG
    s2 = b2.run(prompt, G, pid)
    baseline_records['b2'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': len(s2.get('entities',[])), 'relations': len(s2.get('relations',[])),
        'baseline': 'B2', 'status': 'ok'})

    # B3: IFC direct lookup
    s3 = b3.run(prompt, G, ifc_config, pid)
    baseline_records['b3'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': len(s3.get('entities',[])), 'relations': len(s3.get('relations',[])),
        'baseline': 'B3', 'status': 'ok'})

    print(f'[{i+1:02d}/{len(all_prompts)}] {pid} B0:0e B2:{len(s2.get("entities",[]))}e B3:{len(s3.get("entities",[]))}e', end='\r')

print(f'\n✓ CPU baselines complete.')
for b, recs in baseline_records.items():
    ok = sum(1 for r in recs if r['status'] == 'ok')
    print(f'  {b.upper()}: {ok}/{len(recs)} OK')

[50/50] T3-STR-010 B0:0e B2:10e B3:8e
✓ CPU baselines complete.
  B0: 50/50 OK
  B1: 50/50 OK
  B2: 50/50 OK
  B3: 50/50 OK


In [5]:
# ── Cell 5: Run B4 — IFC-GraphRAG-DT ─────────────────────────────────────────
HAS_GPU = torch.cuda.is_available()
print(f'GPU: {HAS_GPU}')

from pipeline.run_pipeline import IFCGraphRAGDT
pipeline = IFCGraphRAGDT(
    config_path='pipeline/configs/pipeline_config.yaml',
    dry_run=not HAS_GPU,  # skip 3D generation on CPU
    offline=(SPEC_PROVIDER == 'deterministic')
)
print(f'Pipeline mode: {"FULL (GPU)" if HAS_GPU else "DRY RUN (no 3D gen)"}')

b4_records = []
for i, p in enumerate(all_prompts):
    pid, prompt, tier = p['id'], p['prompt'], p['tier']
    t0 = time.time()
    try:
        result = pipeline.run(prompt=prompt, prompt_id=pid, tier=tier)
        b4_records.append({'prompt_id': pid, 'tier': tier, 'domain': p.get('domain','MEP'),
            'entities': len(result.spec.get('entities',[])),
            'relations': len(result.spec.get('relations',[])),
            'mesh_path': str(result.mesh_path) if result.mesh_path else None,
            'elapsed_s': round(time.time()-t0, 2), 'baseline': 'B4', 'status': 'ok'})
    except Exception as e:
        b4_records.append({'prompt_id': pid, 'tier': tier, 'domain': p.get('domain','MEP'),
            'entities': 0, 'relations': 0, 'mesh_path': None,
            'elapsed_s': round(time.time()-t0, 2), 'baseline': 'B4', 'status': f'error: {e}'})
    print(f'[{i+1:02d}/{len(all_prompts)}] {pid} — {round(time.time()-t0,1)}s', end='\r')

ok = sum(1 for r in b4_records if r['status'] == 'ok')
print(f'\n✓ B4 complete: {ok}/{len(b4_records)} OK')

GPU: False
Pipeline mode: DRY RUN (no 3D gen)
2026-06-09 10:06:51,373 [INFO] pipeline.run_pipeline: ============================================================


2026-06-09 10:06:51,374 [INFO] pipeline.run_pipeline: Pipeline run: [T1-MEP-001] Generate a centrifugal pump


2026-06-09 10:06:51,375 [INFO] pipeline.run_pipeline: Loading cached IFC graph from outputs\graphs\ifc_graph.json


2026-06-09 10:06:51,389 [INFO] pipeline.layer1_retriever.graph_embedder: Loading embedding model sentence-transformers/all-MiniLM-L6-v2 on cpu


2026-06-09 10:06:51,661 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


...

2026-06-09 10:06:59,700 [INFO] pipeline.run_pipeline: Layer 1: 179 nodes, 256 edges retrieved


2026-06-09 10:06:59,704 [INFO] pipeline.layer2_spec_gen.scene_spec_generator: Generating spec for [T3-STR-010]: Generate a long-span steel truss bridge with two parallel ma


2026-06-09 10:06:59,725 [INFO] pipeline.run_pipeline: Layer 2: 179 entities, 256 relations in spec


2026-06-09 10:06:59,725 [INFO] pipeline.run_pipeline: Layer 3: skipped (dry_run=True)


[50/50] T3-STR-010 — 0.1s
✓ B4 complete: 50/50 OK


In [6]:
# ── Cell 6: Save all results ──────────────────────────────────────────────────
all_records = {**baseline_records, 'b4': b4_records}

for b_name, recs in all_records.items():
    out = f'outputs/scores/{b_name}_scores.json'
    with open(out, 'w') as f:
        json.dump(recs, f, indent=2)
    print(f'Saved: {out} ({len(recs)} records)')

# Copy to Drive
if DRIVE_OUT != 'outputs':
    import shutil
    shutil.copytree('outputs/scores', f'{DRIVE_OUT}/scores', dirs_exist_ok=True)
    if HAS_GPU:
        shutil.copytree('outputs/meshes', f'{DRIVE_OUT}/meshes', dirs_exist_ok=True)
    print(f'Results copied to Google Drive.')

Saved: outputs/scores/b0_scores.json (50 records)

Saved: outputs/scores/b1_scores.json (50 records)
Saved: outputs/scores/b2_scores.json (50 records)
Saved: outputs/scores/b3_scores.json (50 records)
Saved: outputs/scores/b4_scores.json (50 records)


In [7]:
# ── Cell 7: Quick summary table ───────────────────────────────────────────────
print('ENTITY COUNT SUMMARY (mean per prompt, proxy for retrieval quality)')
print(f'{"Baseline":<8} {"Tier 1":>10} {"Tier 2":>10} {"Tier 3":>10}')
print('-' * 42)
for b_name, recs in all_records.items():
    means = []
    for tier in [1, 2, 3]:
        t_recs = [r for r in recs if r.get('tier')==tier and r.get('status')=='ok']
        means.append(np.mean([r.get('entities',0) for r in t_recs]) if t_recs else 0.0)
    print(f'{b_name.upper():<8} {means[0]:>10.1f} {means[1]:>10.1f} {means[2]:>10.1f}')
print('\nProceed to Notebook 05 for evaluation and paper figures.')

ENTITY COUNT SUMMARY (mean per prompt, proxy for retrieval quality)
Baseline     Tier 1     Tier 2     Tier 3
------------------------------------------
B0              0.0        0.0        0.0
B1              0.9        1.5        1.8
B2             10.0       10.0       10.0
B3              0.0        0.0        7.2
B4             11.6       90.3      229.0

Proceed to Notebook 05 for evaluation and paper figures.
